In [ ]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

client = OpenAI(api_key="")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("processed_assessments_clean.json","r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Assessments:",len(df))

Assessments: 377


In [3]:
def build_retrieval_text(row):

    parts = []

    if row.get("description"):
        parts.append(row["description"])

    if isinstance(row.get("assessed_skills_norm"),list):
        parts.append("skills " + " ".join(row["assessed_skills_norm"]))

    if isinstance(row.get("target_roles_norm"),list):
        parts.append("roles " + " ".join(row["target_roles_norm"]))

    if isinstance(row.get("cognitive_dimensions_norm"),list):
        parts.append("competencies " + " ".join(row["cognitive_dimensions_norm"]))

    return " ".join(parts)

df["retrieval_text"] = df.apply(build_retrieval_text,axis=1)

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
embeddings = model.encode(
    df["retrieval_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 12/12 [00:07<00:00,  1.67it/s]


In [6]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(df["retrieval_text"])
tfidf_matrix = normalize(tfidf_matrix)

In [21]:
def reason_query(query):

    prompt = f"""
You are an expert HR assessment consultant.

Analyze the hiring query or job description.

Extract structured hiring requirements.

Return JSON with fields:

role
seniority
experience_years
technical_skills
competencies
assessment_types
max_duration

experience_years:
Estimated required experience in years if mentioned.

Query:
{query}

Return JSON only.
"""

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    text = res.choices[0].message.content

    m = re.search(r"\{.*\}",text,re.DOTALL)

    if m:
        return json.loads(m.group())

    return {}

In [8]:
def detect_intent(parsed):

    skills = parsed.get("technical_skills",[])
    role = parsed.get("role","")
    comps = parsed.get("competencies",[])

    if skills:
        return "skill"

    if role:
        return "role"

    if any(c in ["leadership","culture","communication"] for c in comps):
        return "leadership"

    return "general"

In [9]:
def build_query_text(parsed):

    parts = []

    if parsed.get("role"):
        parts.append(parsed["role"])

    parts += parsed.get("technical_skills",[])
    parts += parsed.get("competencies",[])

    return " ".join(parts)

In [10]:
def retrieve_candidates(query_text,top_k=100):

    q_emb = model.encode(query_text,normalize_embeddings=True)

    sims = cosine_similarity([q_emb],embeddings)[0]

    idx = np.argsort(sims)[::-1][:top_k]

    return idx,sims

In [11]:
def adaptive_score(parsed,intent,idx,sims,query_text):

    q_tfidf = tfidf.transform([query_text])
    tfidf_scores = (tfidf_matrix[idx] @ q_tfidf.T).toarray().ravel()

    role_text = parsed.get("role","")
    skill_text = " ".join(parsed.get("technical_skills",[]))
    comp_text = " ".join(parsed.get("competencies",[]))

    role_emb = model.encode(role_text,normalize_embeddings=True) if role_text else None
    skill_emb = model.encode(skill_text,normalize_embeddings=True) if skill_text else None
    comp_emb = model.encode(comp_text,normalize_embeddings=True) if comp_text else None

    role_sim = cosine_similarity([role_emb],embeddings[idx])[0] if role_emb is not None else np.zeros(len(idx))
    skill_sim = cosine_similarity([skill_emb],embeddings[idx])[0] if skill_emb is not None else np.zeros(len(idx))
    comp_sim = cosine_similarity([comp_emb],embeddings[idx])[0] if comp_emb is not None else np.zeros(len(idx))

    if intent=="skill":

        score = (
            0.5*skill_sim +
            0.25*sims[idx] +
            0.15*tfidf_scores +
            0.10*comp_sim
        )

    elif intent=="role":

        score = (
            0.5*role_sim +
            0.25*sims[idx] +
            0.15*tfidf_scores +
            0.10*comp_sim
        )

    elif intent=="leadership":

        score = (
            0.5*comp_sim +
            0.25*role_sim +
            0.15*sims[idx] +
            0.10*tfidf_scores
        )

    else:

        score = (
            0.4*sims[idx] +
            0.3*tfidf_scores +
            0.3*comp_sim
        )

    return score

In [22]:
def experience_boost(df_subset, query_exp):

    if query_exp is None:
        return np.zeros(len(df_subset))

    boosts = []

    for exp in df_subset.get("experience_numeric", []):

        try:
            midpoint = exp.get("midpoint", None)
        except:
            midpoint = None

        if midpoint is None:
            boosts.append(0)
            continue

        dist = abs(midpoint - query_exp)

        # closer experience gets higher score
        score = max(0, 0.1 - 0.02*dist)

        boosts.append(score)

    return np.array(boosts)

In [13]:
def ensure_coverage(results,competencies):

    if not competencies:
        return results.head(10)

    selected = []
    covered = set()

    for idx,row in results.iterrows():

        text = row["retrieval_text"].lower()

        match = [c for c in competencies if c.lower() in text]

        if match:

            selected.append(idx)
            covered.update(match)

        if len(selected)>=10:
            break

    for idx in results.index:

        if idx not in selected:

            selected.append(idx)

        if len(selected)>=10:
            break

    return results.loc[selected].head(10)

In [24]:
def recommend(query):

    parsed = reason_query(query)

    intent = detect_intent(parsed)

    query_text = build_query_text(parsed)

    idx,sims = retrieve_candidates(query_text)

    scores = adaptive_score(parsed,intent,idx,sims,query_text)

    res = df.iloc[idx].copy()

    res["score"] = scores

    exp_boost = experience_boost(res, parsed.get("experience_years"))

    res["score"] += exp_boost

    res = res.sort_values("score",ascending=False)

    res = ensure_coverage(res,parsed.get("competencies",[]))

    return res

In [15]:
def slug(url):

    if not isinstance(url,str):
        return ""

    url = url.lower()

    m = re.search(r"/view/([^/]+)",url)

    if m:
        return m.group(1)

    return url

In [16]:
def recall_at_10(pred,true):

    p = {slug(u) for u in pred}
    t = {slug(u) for u in true}

    if not t:
        return 0

    return len(p & t)/len(t)

In [ ]:
eval_df = pd.read_excel("Gen_AI Dataset.xlsx",sheet_name="Train-Set")

grouped = eval_df.groupby("Query")["Assessment_url"].apply(list).to_dict()

In [18]:
def evaluate():

    recalls=[]

    for q,urls in tqdm(grouped.items()):

        res = recommend(q)

        preds = res["url"].tolist()

        r = recall_at_10(preds,urls)

        recalls.append(r)

    return np.mean(recalls)

In [25]:
mean_recall = evaluate()

print("Mean Recall@10:",mean_recall)

100%|██████████| 10/10 [00:29<00:00,  2.92s/it]

Mean Recall@10: 0.29


In [26]:
rows=[]

for q in grouped:

    res = recommend(q)

    for rank,row in enumerate(res.itertuples(),1):

        rows.append({
            "query":q,
            "rank":rank,
            "name":row.name,
            "url":row.url,
            "score":row.score
        })

pd.DataFrame(rows).to_excel("recommendations_approach4.xlsx",index=False)

In [27]:
rows = []

for query, true_urls in grouped.items():

    # Convert ground truth URLs to slugs
    true_slugs = {slug(u) for u in true_urls}

    # Get recommendations
    res = recommend(query)

    for rank, row in enumerate(res.itertuples(), start=1):

        pred_slug = slug(row.url)

        rows.append({
            "query": query,
            "rank": rank,
            "assessment_name": row.name,
            "predicted_url": row.url,
            "score": row.score,
            "predicted_slug": pred_slug,
            "is_relevant": pred_slug in true_slugs,
            "ground_truth_urls": ", ".join(true_urls)
        })

df_compare = pd.DataFrame(rows)

df_compare.head()

,query,rank,assessment_name,predicted_url,score,predicted_slug,is_relevant,ground_truth_urls
0,Based on the JD below recommend me assessment ...,1,Time Management (U.S.),https://www.shl.com/products/product-catalog/v...,0.450494,time-management-u-s,False,https://www.shl.com/solutions/products/product...
1,Based on the JD below recommend me assessment ...,2,Managerial Scenarios Profile Report,https://www.shl.com/products/product-catalog/v...,0.442019,managerial-scenarios-profile-report,False,https://www.shl.com/solutions/products/product...
2,Based on the JD below recommend me assessment ...,3,Managerial Scenarios Narrative Report,https://www.shl.com/products/product-catalog/v...,0.442019,managerial-scenarios-narrative-report,False,https://www.shl.com/solutions/products/product...
3,Based on the JD below recommend me assessment ...,4,Managerial Scenarios Candidate Report,https://www.shl.com/products/product-catalog/v...,0.442019,managerial-scenarios-candidate-report,False,https://www.shl.com/solutions/products/product...
4,Based on the JD below recommend me assessment ...,5,OPQ Universal Competency Report 1.0,https://www.shl.com/products/product-catalog/v...,0.367979,opq-universal-competency-report,False,https://www.shl.com/solutions/products/product...


In [29]:
df_compare.to_excel("recommendation_comparison_4_1.xlsx", index=False)

print("Saved to recommendation_comparison.xlsx")

Saved to recommendation_comparison.xlsx


In [30]:
query_hits = df_compare.groupby("query")["is_relevant"].sum().reset_index()

query_hits.columns = ["query", "correct_predictions"]

query_hits

,query,correct_predictions
0,Based on the JD below recommend me assessment ...,0
1,"Content Writer required, expert in English and...",2
2,Find me 1 hour long assesment for the below jo...,5
3,I am hiring for Java developers who can also c...,4
4,I am looking for a COO for my company in China...,1
5,I want to hire a Senior Data Analyst with 5 ye...,5
6,I want to hire new graduates for a sales role ...,1
7,"ICICI Bank Assistant Admin, Experience require...",1
8,KEY RESPONSIBITILES:\n\nManage the sound-scape...,0
9,We're looking for a Marketing Manager who can ...,2
